<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_1_GEE_Forest_Area_Computation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 1 — GEE Forest Area Computation
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [ ]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q earthengine-api geemap

In [ ]:
# ── clone repo if not already cloned ─────────────────────
import os
import getpass
import subprocess

if not os.path.exists('/content/Biochar_forest_estimation'):
    PAT = getpass.getpass('Enter PAT: ')
    !git config --global user.email "sdkmajd@gmail.com"
    !git config --global user.name "MElsdk-lab"
    subprocess.run(
        f'git clone https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git',
        shell=True
    )

%cd /content/Biochar_forest_estimation/
!git fetch origin
!git reset --hard origin/main
print('✅ Ready')

Enter PAT: ··········
/content/Biochar_forest_estimation
HEAD is now at 8e12148 run test
✅ Ready


In [ ]:
import ee
import geemap
from data_config import FAO_LSIB_REGION, us_state_names, get_all_countries, build_country_lookup, country_thresholds, state_thresholds

print('✅ Libraries imported')
print('✅ Data config loaded')

✅ Libraries imported
✅ Data config loaded


In [13]:
import ee

# Authenticate and initialize Earth Engine
try:
    ee.Initialize(project='spry-blade-487218-n0')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='spry-blade-487218-n0')  # ← add project here too


# ── CELL 3: Load Datasets ────────────────────────────────────────
Hansen_GFC_2024      = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GLC_FSC30D_annual    = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/annual')
GLC_FSC30D_five_year = ee.ImageCollection('projects/sat-io/open-datasets/GLC-FCS30D/five-years-map')

print('✅ Hansen GFC 2024 loaded')
print('✅ GLC FCS30D annual loaded')
print('✅ GLC FCS30D five-year loaded')


✅ Hansen GFC 2024 loaded
✅ GLC FCS30D annual loaded
✅ GLC FCS30D five-year loaded


In [ ]:
# ── CELL 4: Select & Mask Hansen Bands ───────────────────────────────────────
treecover2000 = Hansen_GFC_2024.select('treecover2000')
datamask      = Hansen_GFC_2024.select('datamask')

treecover2000_masked = treecover2000.updateMask(treecover2000.gt(0)) \
                                    .updateMask(datamask.eq(1))

print('✅ treecover2000 masked')

# ── CELL 5: Load GLC FCS30D Year 2000 ────────────────────────────────────────
glc_2000        = GLC_FSC30D_annual.mosaic().select('b1')
glc_2000_forest = glc_2000.gte(51).And(glc_2000.lte(92)).selfMask()

print('✅ GLC FCS30D 2000 loaded')
print('✅ GLC forest mask created (classes 51-92)')

# ── CELL 6: Forest Class Definitions ─────────────────────────────────────────
forestClasses = [
    {'code': 51, 'color': '4c7300', 'name': '51 Open evergreen broadleaved'},
    {'code': 52, 'color': '006400', 'name': '52 Closed evergreen broadleaved'},
    {'code': 61, 'color': 'a8c800', 'name': '61 Open deciduous broadleaved'},
    {'code': 62, 'color': '00a000', 'name': '62 Closed deciduous broadleaved'},
    {'code': 71, 'color': '005000', 'name': '71 Open evergreen needleleaved'},
    {'code': 72, 'color': '003c00', 'name': '72 Closed evergreen needleleaved'},
    {'code': 81, 'color': '286400', 'name': '81 Open deciduous needleleaved'},
    {'code': 82, 'color': '285000', 'name': '82 Closed deciduous needleleaved'},
    {'code': 91, 'color': 'a0b432', 'name': '91 Open mixed forest'},
    {'code': 92, 'color': '788200', 'name': '92 Closed mixed forest'},
]

print(f'✅ {len(forestClasses)} forest classes defined')

NameError: name 'Hansen_GFC_2024' is not defined

In [ ]:
# ── CELL 9: GEE Functions  — Countries ────────────────────────────────────────
def prepare_forest_collection(selected_regions, threshold):
    """
    Prepare a GEE FeatureCollection with forest area per country
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    all_countries = get_all_countries(selected_regions)

    lsib_fao = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
                 .filter(ee.Filter.inList('country_na', all_countries))

    forest_mask = treecover2000_masked.gte(threshold) \
                                      .selfMask() \
                                      .updateMask(datamask.eq(1))

    area_image = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

    region_areas = area_image.reduceRegions(
        collection=lsib_fao,
        reducer=ee.Reducer.sum(),
        scale=30,
    )

    region_areas = region_areas.map(lambda f: f.set('threshold', threshold))

    return region_areas


def export_forest_area(selected_regions, thresholds):
    """
    Submit one GEE export task per threshold to Google Drive.
    Returns list of tasks for monitoring.
    """
    tasks = []

    for threshold in thresholds:
        fc = prepare_forest_collection(selected_regions, threshold)

        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'forest_area_{threshold}',
            folder='/content/drive/MyDrive/BiocharProject/GEE_exports',
            fileNamePrefix=f'forest_area_{threshold}',
            fileFormat='CSV',
            selectors=['country_na', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'✅ Task submitted for threshold {threshold}%')

    return tasks


print('✅ prepare_forest_collection() defined')
print('✅ export_forest_area() defined')

In [ ]:
# ── CELL 10: GEE Functions — US States ───────────────────────────────────────
tiger_states = ee.FeatureCollection('TIGER/2018/States') \
                 .filter(ee.Filter.inList('NAME', us_state_names))


def prepare_states_forest_collection(threshold):
    """
    Prepare a GEE FeatureCollection with forest area per US state
    for a given canopy cover threshold.
    Returns a GEE object (not yet computed).
    """
    forest_mask = treecover2000_masked.gte(threshold) \
                                      .selfMask() \
                                      .updateMask(datamask.eq(1))

    area_image = forest_mask.multiply(ee.Image.pixelArea().divide(1e10))

    states_areas = area_image.reduceRegions(
        collection=tiger_states,
        reducer=ee.Reducer.sum(),
        scale=30,
        maxPixelsPerRegion=1e13
    )

    states_areas = states_areas.map(lambda f: f.set('threshold', threshold))

    return states_areas


def export_states_forest_area(thresholds):
    """
    Submit one GEE export task per threshold to Google Drive.
    Returns list of tasks for monitoring.
    """
    tasks = []

    for threshold in thresholds:
        fc = prepare_states_forest_collection(threshold)

        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'states_forest_area_{threshold}',
            folder='/content/drive/MyDrive/BiocharProject/GEE_exports',
            fileNamePrefix=f'states_forest_area_{threshold}',
            fileFormat='CSV',
            selectors=['NAME', 'threshold', 'sum']
        )
        task.start()
        tasks.append(task)
        print(f'✅ Task submitted for threshold {threshold}%')

    return tasks


print('✅ prepare_states_forest_collection() defined')
print('✅ export_states_forest_area() defined')
print(f'✅ {len(us_state_names)} US states defined')

In [ ]:
# ── CELL 11: Run Exports ──────────────────────────────────────────────────────

print('── Submitting country tasks ──')
country_tasks = export_forest_area(FAO_LSIB_REGION, country_thresholds)

print('\n── Submitting state tasks ──')
state_tasks = export_states_forest_area(state_thresholds)

In [ ]:
#── CELL 12: Monitor Tasks ────────────────────────────────────────────────────
print('── Country tasks ──')
for task in country_tasks:
    status = task.status()
    print(f"  {status['description']}: {status['state']}")

print('\n── State tasks ──')
for task in state_tasks:
    status = task.status()
    print(f"  {status['description']}: {status['state']}")

In [ ]:
# ── CELL 13: export GEE exports created from Notebook 1 and stored on Google drive to github────────────────────────────────────────
import shutil
print(os.listdir(GEE_FOLDER))

# copy raw GEE exports from Drive to repo data folder
files = [
    'forest_area_10.csv',    'forest_area_20.csv', 'forest_area_30.csv','forest_area_40.csv',  'forest_area_50.csv',
    'states_forest_area_10.csv', 'states_forest_area_20.csv','states_forest_area_30.csv', 'states_forest_area_40.csv',  'states_forest_area_50.csv'
        ]

for f in files:
    shutil.copy(GEE_FOLDER + f, DATA_FOLDER + f)
    print(f'✅ Copied {f} to repo')

In [ ]:
# ── push results to GitHub ────────────────────────────────
import subprocess

# ask for PAT only if not already defined
if 'PAT' not in dir():
    import getpass
    PAT = getpass.getpass('Enter PAT: ')

%cd /content/Biochar_forest_estimation/
!git add .
!git commit -m "update results"

result = subprocess.run(
    f'git push https://{PAT}@github.com/MElsdk-lab/Biochar_forest_estimation.git main',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print('✅ Pushed to GitHub')
else:
    print('❌ Push failed')
    print(result.stderr)